# GROUP PROJECT

**IMPORTANT from classes**

* try remove shit data: create 3 folds and record on what photos model fails - they are the condidats to be removed
* try ebmedinng model and find outliers in that space for each cluster
* mb try base models from hugging face 

* use tensorboard
* add a heatmap on top of image at the end
* plot the closest images

## Importing libraries and setting up google drive

In [ ]:
#
import numpy as np
import pandas as pd

# Data visualisation
import matplotlib.pyplot as plt
from matplotlib import style
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# sklearn related
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight

#
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.regularizers import L1, L2
#
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from keras.layers import Input, Dense, Dropout
from keras.optimizers import Adam, SGD, Adagrad, Adadelta, RMSprop
from keras.utils import to_categorical

# cnn related
from keras.layers import Conv2D, Flatten, MaxPooling2D, GlobalAveragePooling2D, BatchNormalization

from PIL import Image



# sort later :3
import tensorflow as tf
import math
from collections import Counter
from PIL import Image as PilImage

# manipulating files
import cv2
from tqdm import tqdm
import os
import ast

# For image showing
from IPython.display import Image, display
import matplotlib.pyplot as plt
import os


In [ ]:
directory = 'rare_species/'

In [ ]:
# Place this at the very top of your script/notebook
tf.config.optimizer.set_jit(True)
# Alternatively, compile your model with the jit_compile flag
# model.compile(jit_compile=True, ...)

In [ ]:
# Enable Mixed Precision policy at the start
from tensorflow.keras.mixed_precision import set_global_policy
set_global_policy('mixed_float16')
# Make sure the final output layer is still float32 for stability
# final_layer = Dense(num_classes, activation='softmax', dtype='float32')(previous_layer)

In [ ]:
# # Set up google drive FOR COLAB
# from google.colab.patches import cv2_imshow
# from google.colab import drive
# drive.mount('/content/drive/')

# # The directory where all the files will go
# !unzip -o "/content/drive/MyDrive/DeepLearningProject/rare_species.zip" -d {directory} > /dev/null # remove all the prints (takes +-2min yo run)

In [ ]:
# import torch
# print("CUDA available:", torch.cuda.is_available())
# print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

In [ ]:
print("TensorFlow version:", tf.__version__)
print("GPUs detected:", tf.config.list_physical_devices('GPU'))

In [ ]:
# tf.debugging.set_log_device_placement(True)     # to see which device is being used

In [ ]:
metadata = pd.read_csv(f'{directory}metadata.csv')

## Custom functions (later to .py)

In [ ]:
def explore_image_files(file_paths, explore_values=False):
    # Define the variables of smalles and biggest images
    image_sizes = []
    color_channels = []
    formats = []
    min_vals = []
    max_vals = []
    ratios = []

    # Iteration loop for each folder to compare the image sizes

    for file_path in file_paths:
        with PilImage.open(file_path) as img:
            image_sizes.append(img.size)
            color_channels.append(img.mode)
            formats.append(img.format)
            ratios.append(img.size[0]/img.size[1])

            if explore_values:
                # Convert to numpy to check the actual data type, Takes alot of time
                img_array = np.array(img)

                # Value range
                min_vals.append(img_array.min())
                max_vals.append(img_array.max())

    if explore_values:
        return image_sizes, color_channels, formats, ratios, min_vals, max_vals
    else:
        return image_sizes, color_channels, formats, ratios

## Initial exploration

### Metadata exploration

#### Basic exploration

In [ ]:
metadata.info()

In [ ]:
metadata.head()

In [ ]:
metadata.describe(include='O')

In [ ]:
# Get phylum counts
phylum_counts = metadata['phylum'].value_counts()

# Calculate number of families per phylum
families_per_phylum = metadata.groupby('phylum')['family'].nunique()

# Create custom hover data with family counts
hover_data = []
for phylum in phylum_counts.index:
    image_count = phylum_counts[phylum]
    family_count = families_per_phylum[phylum]
    percentage = (image_count / len(metadata)) * 100
    hover_data.append([image_count, percentage, family_count])

# Convert to numpy array for easy indexing
hover_data = np.array(hover_data)

# Create figure
fig = go.Figure(go.Bar(
    x=phylum_counts.index,
    y=phylum_counts.values,
    marker=dict(
        color='#6366f1',
        line=dict(color='#4f46e5', width=0.5)
    ),
    text=phylum_counts.values,
    textposition='outside',
    textfont=dict(size=12),
    hovertemplate='<b>%{x}</b><br>' +
                  'Images: %{y}<br>' +
                  'Families: %{customdata[2]}<br>' +
                  'Percentage: %{customdata[1]:.2f}%<extra></extra>',
    customdata=hover_data
))

fig.update_layout(
    title={
        'text': '<b>Distribution of Species by Phylum</b>',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 18}
    },
    xaxis_title='Phylum',
    yaxis_title='Count',
    height=600,
    width=1000,
    plot_bgcolor='white',
    paper_bgcolor='white',
    xaxis=dict(
        tickfont=dict(size=11),
        showgrid=False
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor='#e5e7eb',
        gridwidth=1
    ),
    font=dict(size=11),
    showlegend=False
)

fig.show()

# Print summary
print("\n" + "="*60)
print("PHYLUM DISTRIBUTION WITH FAMILY COUNTS")
print("="*60)
for phylum in phylum_counts.index:
    image_count = phylum_counts[phylum]
    family_count = families_per_phylum[phylum]
    pct = (image_count / len(metadata)) * 100
    avg_images_per_family = image_count / family_count
    print(f"{phylum:20s}: {image_count:4d} images ({pct:5.2f}%) | {family_count:3d} families | Avg: {avg_images_per_family:.1f} images/family")
print("="*60)

In [ ]:
# Get all family counts
family_counts = metadata['family'].value_counts()

# Create figure with scrollable y-axis
fig = go.Figure(go.Bar(
    x=family_counts.values,
    y=family_counts.index,
    orientation='h',
    marker=dict(
        color='#6366f1',
        line=dict(color='#4f46e5', width=0.5)
    ),
    text=family_counts.values,
    textposition='outside',
    textfont=dict(size=10),
    hovertemplate='<b>%{y}</b><br>Images: %{x}<br>Percentage: %{customdata:.2f}%<extra></extra>',
    customdata=(family_counts.values / len(metadata)) * 100
))

fig.update_layout(
    title={
        'text': '<b>Distribution Of Species By Family</b><br><sub>All families shown - scroll to explore</sub>',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 18}
    },
    xaxis_title='Number of Images',
    yaxis_title='Family',
    height=max(1000, len(family_counts) * 15),  # Dynamic height based on number of families
    width=1200,
    plot_bgcolor='white',
    paper_bgcolor='white',
    yaxis=dict(
        autorange='reversed',  # Highest count on top
        tickfont=dict(size=9),
        showgrid=False
    ),
    xaxis=dict(
        showgrid=True,
        gridcolor='#e5e7eb',
        gridwidth=1
    ),
    font=dict(size=11),
    margin=dict(l=200, r=100, t=100, b=50),  # More space for family names
    showlegend=False
)

fig.show()


In [ ]:

# Count at each taxonomic level
phylum_to_family = metadata.groupby(['phylum', 'family']).size().reset_index(name='count')

# Prepare data for Sankey
labels = list(metadata['phylum'].unique()) + list(metadata['family'].unique())
label_dict = {label: idx for idx, label in enumerate(labels)}

source = []
target = []
value = []

for _, row in phylum_to_family.iterrows():
    source.append(label_dict[row['phylum']])
    target.append(label_dict[row['family']])
    value.append(row['count'])

# Create Sankey diagram
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=labels,
        color='#6366f1'
    ),
    link=dict(
        source=source,
        target=target,
        value=value,
        color='rgba(99, 102, 241, 0.3)'
    )
)])

fig.update_layout(
    title='<b>Taxonomic Hierarchy: Phylum → Family</b>',
    font=dict(size=12),
    height=800,
    width=1200
)

fig.show()

print("\n📊 TAXONOMIC DIVERSITY:")
print(f"Phylums: {metadata['phylum'].nunique()}")
print(f"Families: {metadata['family'].nunique()}")
print(f"Average families per phylum: {metadata['family'].nunique() / metadata['phylum'].nunique():.1f}")

#### Retracting more information from the images

In [ ]:
# # Extract more information about the images and save to the metadata df (6min)
# metadata['image_size'], metadata['color_channel'], metadata['format'], metadata['aspect_ratio'], \
# metadata['min_val'], metadata['max_val'] = explore_image_files(directory + metadata['file_path'], explore_values=True)

# Don't explore values to save time (1min)
metadata['image_size'], metadata['color_channel'], metadata['format'], metadata['aspect_ratio'] = explore_image_files(directory + metadata['file_path'])

In [ ]:
metadata['width'], metadata['height'] = zip(*metadata['image_size'])

In [ ]:
metadata.describe()

huge file at "\rare_species\mollusca_cardiidae\30003931_46473744_eol-full-size-copy.jpg"

In [ ]:
# Show largest and smallest files
image_files = []
for root, dirs, files in os.walk(directory):
    for file in files:
        if file.lower().endswith(('.jpg', '.jpeg', '.png')):
            filepath = os.path.join(root, file)
            try:
                img = PilImage.open(filepath)
                pixel_count = img.size[0] * img.size[1]  # width * height
                image_files.append((filepath, pixel_count, img.size))
                img.close()
            except:
                pass

# Sort by pixel count
if image_files:
    image_files.sort(key=lambda x: x[1], reverse=True)
    largest_image_path, largest_pixels, largest_dims = image_files[0]
    smallest_image_path, smallest_pixels, smallest_dims = image_files[-1]
    
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    
    # Display largest image
    img_largest = PilImage.open(largest_image_path)
    axes[0].imshow(img_largest)
    axes[0].axis('off')
    axes[0].set_title(f"LARGEST IMAGE\n{os.path.basename(largest_image_path)}\n{largest_dims[0]} x {largest_dims[1]} pixels\n({largest_pixels:,} total pixels)", fontsize=12, fontweight='bold')
    
    # Display smallest image
    img_smallest = PilImage.open(smallest_image_path)
    axes[1].imshow(img_smallest)
    axes[1].axis('off')
    axes[1].set_title(f"SMALLEST IMAGE\n{os.path.basename(smallest_image_path)}\n{smallest_dims[0]} x {smallest_dims[1]} pixels\n({smallest_pixels:,} total pixels)", fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print(f"Largest:  {largest_image_path}")
    print(f"          {largest_dims[0]} x {largest_dims[1]} = {largest_pixels:,} pixels")
    print(f"\nSmallest: {smallest_image_path}")
    print(f"          {smallest_dims[0]} x {smallest_dims[1]} = {smallest_pixels:,} pixels")
else:
    print("Error")

- We see a normal looking smallest image, but the largest image looks like it could be split into 4 

## Colour exploration

In [ ]:
# Plot colour channel distribution
color_channel_mapping = {
    'L': 'Greyscale',
    'RGB': 'RGB',
    'RGBA': 'RGBA',
    'P': 'Palette',
    'CMYK': 'CMYK',
    '1': 'Binary',
    'LA': 'Greyscale + Alpha'
}

# Get value counts and map to readable names
color_counts = metadata['color_channel'].value_counts()
color_counts.index = color_counts.index.map(lambda x: color_channel_mapping.get(x, x))

# Create figure
fig = go.Figure(go.Bar(
    x=color_counts.index,
    y=color_counts.values,
    marker=dict(
        color='#6366f1',
        line=dict(color='#4f46e5', width=0.5)
    ),
    text=color_counts.values,
    textposition='outside',
    textfont=dict(size=12),
    hovertemplate='<b>%{x}</b><br>Images: %{y}<br>Percentage: %{customdata:.2f}%<extra></extra>',
    customdata=(color_counts.values / len(metadata)) * 100
))

fig.update_layout(
    title={
        'text': '<b>Distribution Of Colour Channels</b>',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 18}
    },
    xaxis_title='Color Channel',
    yaxis_title='Count',
    height=600,
    width=1000,
    plot_bgcolor='white',
    paper_bgcolor='white',
    xaxis=dict(
        tickfont=dict(size=11),
        showgrid=False
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor='#e5e7eb',
        gridwidth=1
    ),
    font=dict(size=11),
    showlegend=False
)

fig.show()

In [ ]:
# Custom lines based on previous discoveries
common_ratio = 4/3
width_max = 500        # MB CHANGE LATER

In [ ]:
# Create figure with custom styling
fig = go.Figure()

# Add scatter plot with better styling
fig.add_trace(go.Scatter(
    x=metadata['width'],
    y=metadata['height'],
    mode='markers',
    marker=dict(
        size=4,
        color='#6366f1',
        opacity=0.4,
        line=dict(width=0)
    ),
    name='Images',
    hovertemplate='<b>Width:</b> %{x}px<br><b>Height:</b> %{y}px<extra></extra>'
))

# Add aspect ratio reference line (4:3)
max_y = metadata['height'].max()
max_x_for_ratio = common_ratio * max_y
fig.add_trace(go.Scatter(
    x=[0, max_x_for_ratio],
    y=[0, max_y],
    mode='lines',
    line=dict(color='#ef4444', width=2.5, dash='dash'),
    name='4:3 Aspect Ratio',
    hoverinfo='skip'
))

# Add width threshold line
fig.add_trace(go.Scatter(
    x=[width_max, width_max],
    y=[0, max_y],
    mode='lines',
    line=dict(color='#10b981', width=2.5, dash='dash'),
    name=f'Max Width ({width_max}px)',
    hoverinfo='skip'
))

# Update layout with legend on the right and bold title
fig.update_layout(
    title={
        'text': '<b>Image Size Distribution Analysis</b>',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 22, 'color': '#1f2937'}
    },
    xaxis=dict(
        title='Width (pixels)',
        showgrid=True,
        gridcolor='#e5e7eb',
        gridwidth=1,
        zeroline=False,
        title_font=dict(size=14, color='#374151')
    ),
    yaxis=dict(
        title='Height (pixels)',
        showgrid=True,
        gridcolor='#e5e7eb',
        gridwidth=1,
        zeroline=False,
        title_font=dict(size=14, color='#374151')
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=1000,
    height=700,
    showlegend=True,
    legend=dict(
        x=1.02,
        y=1,
        xanchor='left',
        yanchor='top',
        bgcolor='rgba(255, 255, 255, 0.95)',
        bordercolor='#d1d5db',
        borderwidth=1,
        font=dict(size=11)
    ),
    hovermode='closest'
)

fig.show()

# Enhanced summary statistics
image_size = (width_max, round(width_max/common_ratio))
print(f'\n{"="*50}')
print(f'📊 RECOMMENDED IMAGE SIZE: {image_size[0]}x{image_size[1]}')
print(f'{"="*50}')
print(f'   Aspect Ratio: {common_ratio:.2f}:1 (4:3)')
print(f'   Total Images: {len(metadata):,}')
print(f'   Width Range: {metadata["width"].min()}-{metadata["width"].max()}px')
print(f'   Height Range: {metadata["height"].min()}-{metadata["height"].max()}px')
print(f'{"="*50}\n')

In [ ]:
# Create 2D histogram/heatmap of image dimensions
fig = px.density_heatmap(
    metadata,
    x='width',
    y='height',
    nbinsx=50,
    nbinsy=50,
    title='<b>Image Dimension Density Heatmap</b>',
    labels={'width': 'Width (pixels)', 'height': 'Height (pixels)'},
    color_continuous_scale='Blues'
)

fig.update_layout(
    width=900,
    height=700,
    plot_bgcolor='white'
)

fig.show()

### Images exploration

#### All images

In [ ]:
# Print 5 example images of each class (+-3min)
for label in os.listdir(directory):
    path = directory + str(label)

    if not os.path.isdir(path):
        print(f"Directory {path} does not exist.")
        continue

    folder_data = os.listdir(path)
    k = 0
    print(f'{label} ({len(folder_data)} images)')

    # Collect image paths
    image_paths = []
    for image_path in folder_data:
        if k < 5:                                               # <-- change how many images per class
            full_path = os.path.join(path, image_path)
            image_paths.append(full_path)
            k += 1

    # Display images
    if image_paths:
        fig, axes = plt.subplots(1, len(image_paths), figsize=(15, 3))
        if len(image_paths) == 1:
            axes = [axes]

        for ax, img_path in zip(axes, image_paths):
            img = PilImage.open(img_path)
            ax.imshow(img)
            ax.axis('off')

        plt.tight_layout()
        plt.show()

From printing some examples of the images that we'll be working with we can see that we have a few:

- X-ray imagaes
- Images of signs
- Paintings
- Text extracts with no images
- Images unrelated to the class
- Maps
- Varying zoom / color / rotations

#### Grayscale VS CMYK

In [ ]:
datagen = ImageDataGenerator(rescale=1./255)

# Get only greyscale images
temp_generator = datagen.flow_from_dataframe(
    dataframe=metadata[metadata.color_channel == 'L'],
    directory=directory,
    x_col='file_path',
    y_col='family',
    target_size=image_size,
    batch_size=32,
    class_mode='categorical',
    shuffle=False   # Not shuffling
)

In [ ]:
# Get class names from the generator
class_names = list(temp_generator.class_indices.keys())

# Determine the total number of batches in the generator
num_batches = int(math.ceil(temp_generator.n / temp_generator.batch_size))

for i in range(num_batches):
    images, labels = next(temp_generator)

    # Plot the images in the current batch
    batch_size_actual = images.shape[0]
    n_cols = min(8, batch_size_actual)                  # <-- set max columns per row
    n_rows = math.ceil(batch_size_actual / n_cols)

    plt.figure(figsize=(3 * n_cols, 3 * n_rows))

    for j in range(batch_size_actual):
        ax = plt.subplot(n_rows, n_cols, j + 1)
        plt.imshow(images[j])

        # Get the index of the highest probability to find the class name
        label_idx = np.argmax(labels[j])
        plt.title(class_names[label_idx], fontsize=9)
        plt.axis("off")

    plt.tight_layout()
    plt.show()

Some images retain at least the shape of the actual animal, but most of them are really bad

In [ ]:
# Do the same for CMYK
temp_generator = datagen.flow_from_dataframe(
    dataframe=metadata[metadata.color_channel == 'CMYK'],
    directory=directory,
    x_col='file_path',
    y_col='family',
    target_size=image_size,
    batch_size=32,
    class_mode='categorical',
    shuffle=False   # Not shuffling
)

In [ ]:
# Get class names from the generator
class_names = list(temp_generator.class_indices.keys())

# Determine the total number of batches in the generator
num_batches = int(math.ceil(temp_generator.n / temp_generator.batch_size))

for i in range(num_batches):
    images, labels = next(temp_generator)

    # Plot the images in the current batch
    batch_size_actual = images.shape[0]
    n_cols = min(8, batch_size_actual)                  # <-- set max columns per row
    n_rows = math.ceil(batch_size_actual / n_cols)

    plt.figure(figsize=(3 * n_cols, 3 * n_rows))

    for j in range(batch_size_actual):
        ax = plt.subplot(n_rows, n_cols, j + 1)
        plt.imshow(images[j])

        # Get the index of the highest probability to find the class name
        label_idx = np.argmax(labels[j])
        plt.title(class_names[label_idx], fontsize=9)
        plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# Changing the cmyk outliers to rgb in the metadata

metadata.loc[metadata['family'].isin(['chordata_emydidae', 'chordata_psittacidae']) & 
             (metadata['color_channel'] == 'CMYK'), 'color_channel'] = 'RGB'

Most of the images apper to be x-rays of sculls of different animals which don't really help identify the animals by the picture, and will more likely only confuse the model. \
(and there are 2 different images for some reason: 2 parrots and a turtle)

In [ ]:
# Delete the generator to free up resources (small impactbut why not)
del temp_generator

## Preprocessing

## Calculate image embeddings and split

In [ ]:
# Remove CMYK and grayscale images from metadata before split
print(f"Original metadata size: {len(metadata)}")
metadata = metadata[~metadata['color_channel'].isin(['CMYK', 'L'])].reset_index(drop=True)
print(f"After removing CMYK/grayscale: {len(metadata)}")

In [ ]:
# TRY TO ONLY USE RGB
# metadata = metadata[metadata.color_channel == 'RGB'].reset_index(drop=True)

In [ ]:
# good_images_df = good_images_df[good_images_df.family != 'formicidae'].reset_index(drop=True)

## Outlier detection / Removal

### Outlier detection using YOLO

In [ ]:
pip install ultralytics
model = YOLO('yolov8n.pt')  # load the model

In [ ]:
from ultralytics import YOLO
import os
from PIL import Image as PilImage

MODEL_ID = 'yolov8s.pt'
CONFIDENCE_THRESHOLD = 0.85
PERSON_CLASS_ID = 0

In [ ]:
def find_images_with_people(metadata, directory='rare_species/', conf_threshold=CONFIDENCE_THRESHOLD):
    """Find images containing people using YOLO."""
    print(f"Loading {MODEL_ID}...")
    model = YOLO(MODEL_ID)
    
    has_person = []
    person_confidence = []
    
    print(f"Scanning for people with confidence > {conf_threshold}...")
    
    for idx, row in tqdm(metadata.iterrows(), total=len(metadata), desc="YOLO person detection"):
        filepath = os.path.join(directory, row['file_path'])
        
        try:
            results = model.predict(
                filepath,
                verbose=False,
                conf=conf_threshold,
                iou=0.5,
                classes=[PERSON_CLASS_ID]
            )
            
            boxes = results[0].boxes if results and results[0].boxes is not None else None
            
            if boxes and len(boxes) > 0:
                has_person.append(True)
                person_confidence.append(float(boxes.conf.max()))
            else:
                has_person.append(False)
                person_confidence.append(0.0)
                
        except Exception as e:
            has_person.append(False)
            person_confidence.append(0.0)
    
    return has_person, person_confidence

In [ ]:
# Run YOLO person detection
metadata['has_person'], metadata['person_confidence'] = find_images_with_people(metadata, directory=directory)

print(f"\nYOLO Results:")
print(f"  Images with people: {metadata['has_person'].sum()}")
print(f"  Images without people: {(~metadata['has_person']).sum()}")

### Outlier detection using CLIP

In [ ]:
from transformers import CLIPProcessor, CLIPModel
import torch

In [ ]:
# Check for GPU/MPS availability
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"\nUsing device: {device}")

# Load CLIP model
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

In [ ]:
# Load CLIP model
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# Define semantic categories - "good" images first, then "bad" noise types
text_prompts = [
    # Good categories (what we WANT to keep) - indices 0-2
    "a photograph of an animal",
    "a wildlife photograph",
    "a photo of an animal in nature",
    
    # Bad categories (what we WANT to filter out) - indices 3+
    "an x-ray image",
    "a medical scan",
    "a drawing or sketch",
    "a map or diagram",
    "text document or book page",
    "a logo or icon",
    "a cooked meal",
    "prepared food",
    "a human",
    "a photo of a person",
    "a portrait of a human face",
    "people in a photograph",
    "a group of people",
    "a scientist or researcher",
]

# Precompute text embeddings
text_inputs = clip_processor(text=text_prompts, return_tensors="pt", padding=True).to(device)
with torch.no_grad():
    text_features = clip_model.get_text_features(**text_inputs)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

In [ ]:
def compute_clip_scores_batch(metadata, directory="rare_species/", batch_size=32):
    """Process all images and compute CLIP semantic scores."""
    all_results = []
    file_paths = metadata['file_path'].tolist()
    
    for i in tqdm(range(0, len(file_paths), batch_size), desc="Computing CLIP scores"):
        batch_paths = file_paths[i:i+batch_size]
        batch_images = []
        batch_indices = []
        
        for j, path in enumerate(batch_paths):
            try:
                full_path = directory + path
                img = PilImage.open(full_path).convert("RGB")
                batch_images.append(img)
                batch_indices.append(i + j)
            except Exception as e:
                all_results.append({
                    'clip_photo_score': None,
                    'clip_noise_score': None,
                    'clip_semantic_quality': None,
                    'clip_best_match': None
                })
        
        if not batch_images:
            continue
        
        image_inputs = clip_processor(images=batch_images, return_tensors="pt", padding=True).to(device)
        
        with torch.no_grad():
            image_features = clip_model.get_image_features(**image_inputs)
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)
            similarities = (image_features @ text_features.T).cpu().numpy()
        
        for sims in similarities:
            photo_score = sims[:3].max()
            noise_score = sims[3:].max()
            all_results.append({
                'clip_photo_score': float(photo_score),
                'clip_noise_score': float(noise_score),
                'clip_semantic_quality': float(photo_score - noise_score),
                'clip_best_match': text_prompts[sims.argmax()]
            })
    
    return pd.DataFrame(all_results)

In [ ]:
# Run CLIP scoring on your metadata
print("Computing CLIP semantic scores...")
clip_results = compute_clip_scores_batch(metadata, directory=directory, batch_size=32)

# Add CLIP columns to metadata
metadata['clip_photo_score'] = clip_results['clip_photo_score']
metadata['clip_noise_score'] = clip_results['clip_noise_score']
metadata['clip_semantic_quality'] = clip_results['clip_semantic_quality']
metadata['clip_best_match'] = clip_results['clip_best_match']

print(f"Processed {len(metadata)} images")

### Imagenet outlier detection

In [ ]:
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.applications.inception_v3 import preprocess_input, decode_predictions

print("\nLoading InceptionV3 model...")
imagenet_model = InceptionV3(weights='imagenet')
print("Model loaded successfully")

In [ ]:
# Run ImageNet animal detection
is_animal_list = []
animal_prob_list = []
top_class_list = []
top_prob_list = []

for idx, row in tqdm(metadata.iterrows(), total=len(metadata), desc="ImageNet animal detection"):
    img_path = os.path.join(directory, row['file_path'])
    is_animal, animal_prob, top_class, top_prob = check_animal_presence(
        img_path, imagenet_model, threshold=0.02
    )
    is_animal_list.append(is_animal)
    animal_prob_list.append(animal_prob)
    top_class_list.append(top_class)
    top_prob_list.append(top_prob)

metadata['has_animal'] = is_animal_list
metadata['animal_confidence'] = animal_prob_list
metadata['predicted_class'] = top_class_list
metadata['prediction_confidence'] = top_prob_list

print(f"\nImageNet Results:")
print(f"  Images with animals: {metadata['has_animal'].sum()}")
print(f"  Images without animals: {(~metadata['has_animal']).sum()}")

### Visualisations / Comparisons

In [ ]:
def show_all_clip_outliers(metadata, directory="rare_species/", n_cols=10):
    """Display ALL CLIP outliers."""
    bad_images = metadata[metadata['clip_semantic_quality'] < 0].copy()
    bad_images = bad_images.sort_values('clip_semantic_quality', ascending=True)
    
    if len(bad_images) == 0:
        print("No CLIP outliers found!")
        return
    
    print(f"Showing ALL {len(bad_images)} CLIP outliers\n")
    print("Category breakdown:")
    print(bad_images['clip_best_match'].value_counts())
    print("\n" + "="*60 + "\n")
    
    n_rows = math.ceil(len(bad_images) / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(2*n_cols, 2.5*n_rows))
    axes = axes.flatten() if n_rows > 1 else [axes] if n_cols == 1 else axes
    
    for idx, (_, row) in enumerate(bad_images.iterrows()):
        try:
            img = PilImage.open(directory + row['file_path'])
            axes[idx].imshow(img)
            axes[idx].set_title(
                f"{row['clip_best_match'].replace('a ', '').replace('an ', '')[:15]}\n"
                f"{row['clip_semantic_quality']:.2f}",
                fontsize=6
            )
        except:
            pass
        axes[idx].axis('off')
    
    for idx in range(len(bad_images), len(axes)):
        axes[idx].axis('off')
    
    plt.suptitle(f"CLIP Outliers ({len(bad_images)} images)", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
def show_yolo_detections(metadata, directory="rare_species/", n_cols=10):
    """Display images where YOLO detected people."""
    person_images = metadata[metadata['has_person'] == True].copy()
    person_images = person_images.sort_values('person_confidence', ascending=False)
    
    if len(person_images) == 0:
        print("No images with people detected!")
        return
    
    print(f"Showing ALL {len(person_images)} images with people detected\n")
    
    n_rows = math.ceil(len(person_images) / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(2*n_cols, 2.5*n_rows))
    axes = axes.flatten() if n_rows > 1 else [axes] if n_cols == 1 else axes
    
    for idx, (_, row) in enumerate(person_images.iterrows()):
        try:
            img = PilImage.open(directory + row['file_path'])
            axes[idx].imshow(img)
            axes[idx].set_title(f"conf: {row['person_confidence']:.2f}", fontsize=6)
        except:
            pass
        axes[idx].axis('off')
    
    for idx in range(len(person_images), len(axes)):
        axes[idx].axis('off')
    
    plt.suptitle(f"YOLO Person Detections ({len(person_images)} images)", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
def show_non_animal_images(metadata, directory="rare_species/", n_cols=10, n_rows=5):
    """Display images where ImageNet didn't detect animals."""
    non_animals = metadata[metadata['has_animal'] == False].copy()
    non_animals = non_animals.sort_values('animal_confidence', ascending=True)
    
    if len(non_animals) == 0:
        print("All images contain animals!")
        return
    
    n_show = min(n_cols * n_rows, len(non_animals))
    print(f"Showing {n_show} of {len(non_animals)} non-animal images\n")
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(2*n_cols, 2.5*n_rows))
    axes = axes.flatten()
    
    for idx, (_, row) in enumerate(non_animals.head(n_show).iterrows()):
        try:
            img = PilImage.open(directory + row['file_path'])
            axes[idx].imshow(img)
            axes[idx].set_title(
                f"{row['predicted_class'][:12]}\n{row['animal_confidence']:.3f}",
                fontsize=6
            )
        except:
            pass
        axes[idx].axis('off')
    
    for idx in range(n_show, len(axes)):
        axes[idx].axis('off')
    
    plt.suptitle(f"ImageNet Non-Animal Images ({len(non_animals)} total)", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
# Show all outliers
print("\n" + "="*60)
print("VISUALIZING OUTLIERS")
print("="*60)

show_all_clip_outliers(metadata, directory=directory)
show_yolo_detections(metadata, directory=directory)
show_non_animal_images(metadata, directory=directory)

### Outlier removal

In [ ]:
print(f"Before filtering: {len(metadata)} images")

outliers = metadata[metadata['clip_semantic_quality'] < 0].copy()
metadata = metadata[metadata['clip_semantic_quality'] >= 0].reset_index(drop=True)

print(f"Removed: {len(outliers)} outliers")
print(f"After filtering: {len(metadata)} images")

### Import / split the dataset

In [ ]:
print(f"\nNumber of families: {metadata['family'].nunique()}")
low_images_class = metadata['family'].value_counts().idxmin()
print(f"Class with fewest images: '{low_images_class}' with {metadata['family'].value_counts().min()} images")

# First split: train+val / test
train_df, test_df = train_test_split(
    metadata,
    test_size=0.1,
    stratify=metadata['family'],
    shuffle=True,
    random_state=42
)

# Second split: train / val
train_df, val_df = train_test_split(
    train_df,
    test_size=0.1,
    stratify=train_df['family'],
    shuffle=True,
    random_state=42
)

# Print the final proportions of the split
total = len(train_df) + len(val_df) + len(test_df)
print(f"\nSplit complete:")
print(f"  Training  : {len(train_df)} ({len(train_df)/total*100:.1f}%)")
print(f"  Validation: {len(val_df)} ({len(val_df)/total*100:.1f}%)")
print(f"  Testing   : {len(test_df)} ({len(test_df)/total*100:.1f}%)")
print(f"  Total     : {total}")

In [ ]:
# Train generator with augmentation
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=25,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.2,
    shear_range=0.15,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

# Validation/Test generator (no augmentation)
val_test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    directory=directory,
    x_col='file_path',
    y_col='family',
    target_size=image_size,
    batch_size=32,
    class_mode='sparse',
    color_mode='rgb',
    shuffle=True
)

val_generator = val_test_datagen.flow_from_dataframe(
    dataframe=val_df,
    directory=directory,
    x_col='file_path',
    y_col='family',
    target_size=image_size,
    batch_size=32,
    class_mode='sparse',
    color_mode='rgb',
    shuffle=False
)

test_generator = val_test_datagen.flow_from_dataframe(
    dataframe=test_df,
    directory=directory,
    x_col='file_path',
    y_col='family',
    target_size=image_size,
    batch_size=32,
    class_mode='sparse',
    color_mode='rgb',
    shuffle=False
)

print(f"\n✅ Data generators created")

In [ ]:
# Get a batch of images and labels
images, labels = next(train_generator)

# Get class names
class_names = list(train_generator.class_indices.keys())

# Plot the images
plt.figure(figsize=(12, 8))

for i in range(min(21, len(images))):
    ax = plt.subplot(3, 7, i + 1)
    plt.imshow(images[i])
    plt.title(class_names[int(labels[i])], fontsize=8)  # sparse labels are integers
    plt.axis("off")

plt.suptitle("Sample Training Batch (with augmentation)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Model


In [ ]:
N_CLASSES = len(train_generator.class_indices)
print(f"Number of classes: {N_CLASSES}")

In [ ]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)
class_weight_dict = dict(enumerate(class_weights))

In [ ]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import (
    Dense, Dropout, GlobalAveragePooling2D, BatchNormalization, 
    Input, Concatenate, LeakyReLU
)
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import L2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print("\n" + "="*60)
print("BUILDING IMAGE MODEL (EfficientNetB0)")
print("="*60)

# Load base model
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(*image_size, 3)
)

# Freeze all layers initially
base_model.trainable = True

# Unfreeze the last N layers (experiment with this number)
# EfficientNetB0 has 237 layers total
UNFREEZE_LAYERS = 20

for layer in base_model.layers[:-UNFREEZE_LAYERS]:
    layer.trainable = False

# Count trainable layers
trainable_count = sum([1 for layer in base_model.layers if layer.trainable])
print(f"Total layers in EfficientNetB0: {len(base_model.layers)}")
print(f"Trainable layers: {trainable_count}")
print(f"Frozen layers: {len(base_model.layers) - trainable_count}")

# Build the image model with custom dense layers
image_input = Input(shape=(*image_size, 3), name='image_input')

x = base_model(image_input)
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)

# Custom dense layers with non-linearity
x = Dense(512, kernel_regularizer=L2(0.01))(x)
x = LeakyReLU(alpha=0.1)(x)
x = Dropout(0.3)(x)

x = Dense(256, kernel_regularizer=L2(0.01))(x)
x = LeakyReLU(alpha=0.1)(x)
x = Dropout(0.2)(x)

x = Dense(128, kernel_regularizer=L2(0.01))(x)
x = LeakyReLU(alpha=0.1)(x)
x = Dropout(0.1)(x)

image_output = Dense(N_CLASSES, activation='softmax', name='image_output')(x)

image_model = Model(inputs=image_input, outputs=image_output, name='image_model')

# Print model summary
print(f"\nImage Model Summary:")
print(f"  Total params: {image_model.count_params():,}")
trainable_params = sum([tf.reduce_prod(w.shape).numpy() for w in image_model.trainable_weights])
non_trainable_params = sum([tf.reduce_prod(w.shape).numpy() for w in image_model.non_trainable_weights])
print(f"  Trainable params: {trainable_params:,}")
print(f"  Non-trainable params: {non_trainable_params:,}")

# Compile with lower learning rate for fine-tuning
image_model.compile(
    optimizer=Adam(learning_rate=0.0001),  # Lower LR for fine-tuning
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Callbacks
image_callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3,
        patience=3,
        min_lr=1e-7,
        verbose=1
    )
]

# Train the image model
print("\nTraining Image Model...")
history_image = image_model.fit(
    train_generator,
    epochs=50,
    class_weight=class_weight_dict,
    validation_data=val_generator,
    callbacks=image_callbacks,
    verbose=1
)

# Plot results
plot_model_results(history_image, 'loss')
plot_model_results(history_image, 'accuracy')


In [ ]:
print("\n" + "="*60)
print("BUILDING METADATA MODEL")
print("="*60)

from sklearn.preprocessing import LabelEncoder

# Prepare metadata features
# Encode categorical variables
phylum_encoder = LabelEncoder()
metadata['phylum_encoded'] = phylum_encoder.fit_transform(metadata['phylum'])

# Get number of unique values
n_phylums = metadata['phylum'].nunique()

print(f"Number of phylums: {n_phylums}")

# Create metadata arrays for train/val/test
def get_metadata_features(df):
    """Extract metadata features for a dataframe."""
    return df['phylum_encoded'].values

train_meta = get_metadata_features(train_df)
val_meta = get_metadata_features(val_df)
test_meta = get_metadata_features(test_df)

# Get labels
train_labels = train_df['family'].map(train_generator.class_indices).values
val_labels = val_df['family'].map(train_generator.class_indices).values
test_labels = test_df['family'].map(train_generator.class_indices).values

print(f"Train metadata shape: {train_meta.shape}")
print(f"Val metadata shape: {val_meta.shape}")
print(f"Test metadata shape: {test_meta.shape}")

# Build metadata model
from tensorflow.keras.layers import Embedding, Flatten

meta_input = Input(shape=(1,), name='meta_input')

# Embedding for phylum (learned representation)
x = Embedding(input_dim=n_phylums, output_dim=32, name='phylum_embedding')(meta_input)
x = Flatten()(x)

x = Dense(64, kernel_regularizer=L2(0.01))(x)
x = LeakyReLU(alpha=0.1)(x)
x = Dropout(0.2)(x)

x = Dense(32, kernel_regularizer=L2(0.01))(x)
x = LeakyReLU(alpha=0.1)(x)

meta_output = Dense(N_CLASSES, activation='softmax', name='meta_output')(x)

metadata_model = Model(inputs=meta_input, outputs=meta_output, name='metadata_model')

metadata_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print(f"\nMetadata Model Summary:")
metadata_model.summary()

# Train metadata model
meta_callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    )
]

print("\nTraining Metadata Model...")
history_meta = metadata_model.fit(
    train_meta,
    train_labels,
    epochs=100,
    batch_size=32,
    class_weight=class_weight_dict,
    validation_data=(val_meta, val_labels),
    callbacks=meta_callbacks,
    verbose=1
)

# Plot results
plot_model_results(history_meta, 'loss')
plot_model_results(history_meta, 'accuracy')

In [ ]:
print("\n" + "="*60)
print("BUILDING ENSEMBLE MODEL")
print("="*60)

# Create a combined model that takes both image and metadata inputs
# and fuses them before the final classification

# Image branch (reuse trained base)
image_input_ensemble = Input(shape=(*image_size, 3), name='image_input_ensemble')
x_img = base_model(image_input_ensemble)
x_img = GlobalAveragePooling2D()(x_img)
x_img = BatchNormalization()(x_img)
x_img = Dense(256, kernel_regularizer=L2(0.01))(x_img)
x_img = LeakyReLU(alpha=0.1)(x_img)
x_img = Dropout(0.2)(x_img)

# Metadata branch
meta_input_ensemble = Input(shape=(1,), name='meta_input_ensemble')
x_meta = Embedding(input_dim=n_phylums, output_dim=32)(meta_input_ensemble)
x_meta = Flatten()(x_meta)
x_meta = Dense(64, kernel_regularizer=L2(0.01))(x_meta)
x_meta = LeakyReLU(alpha=0.1)(x_meta)

# Concatenate both branches
combined = Concatenate()([x_img, x_meta])

# Fusion layers
x = Dense(256, kernel_regularizer=L2(0.01))(combined)
x = LeakyReLU(alpha=0.1)(x)
x = Dropout(0.3)(x)

x = Dense(128, kernel_regularizer=L2(0.01))(x)
x = LeakyReLU(alpha=0.1)(x)
x = Dropout(0.2)(x)

ensemble_output = Dense(N_CLASSES, activation='softmax', name='ensemble_output')(x)

ensemble_model = Model(
    inputs=[image_input_ensemble, meta_input_ensemble],
    outputs=ensemble_output,
    name='ensemble_model'
)

print(f"\nEnsemble Model Summary:")
ensemble_model.summary()

# Compile
ensemble_model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
class EnsembleDataGenerator(tf.keras.utils.Sequence):
    """Custom generator that yields (image, metadata) pairs with labels."""
    
    def __init__(self, dataframe, directory, image_size, batch_size, 
                 class_indices, augment=False, shuffle=True):
        self.dataframe = dataframe.reset_index(drop=True)
        self.directory = directory
        self.image_size = image_size
        self.batch_size = batch_size
        self.class_indices = class_indices
        self.augment = augment
        self.shuffle = shuffle
        self.indices = np.arange(len(self.dataframe))
        
        if self.shuffle:
            np.random.shuffle(self.indices)
    
    def __len__(self):
        return int(np.ceil(len(self.dataframe) / self.batch_size))
    
    def __getitem__(self, idx):
        batch_indices = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        batch_df = self.dataframe.iloc[batch_indices]
        
        # Load images
        images = []
        for _, row in batch_df.iterrows():
            img_path = os.path.join(self.directory, row['file_path'])
            img = tf.keras.preprocessing.image.load_img(img_path, target_size=self.image_size)
            img_array = tf.keras.preprocessing.image.img_to_array(img)
            img_array = preprocess_input(img_array)
            
            # Apply augmentation if training
            if self.augment:
                img_array = self._augment_image(img_array)
            
            images.append(img_array)
        
        images = np.array(images)
        
        # Get metadata
        metadata_features = batch_df['phylum_encoded'].values
        
        # Get labels
        labels = batch_df['family'].map(self.class_indices).values
        
        return [images, metadata_features], labels
    
    def _augment_image(self, img):
        """Apply random augmentations."""
        # Random horizontal flip
        if np.random.random() > 0.5:
            img = np.fliplr(img)
        
        # Random rotation (small angle)
        if np.random.random() > 0.5:
            angle = np.random.uniform(-15, 15)
            img = tf.keras.preprocessing.image.apply_affine_transform(
                img, theta=angle, fill_mode='nearest'
            )
        
        # Random zoom
        if np.random.random() > 0.5:
            zoom = np.random.uniform(0.9, 1.1)
            img = tf.keras.preprocessing.image.apply_affine_transform(
                img, zx=zoom, zy=zoom, fill_mode='nearest'
            )
        
        return img
    
    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)


# Encode phylum for train/val/test dataframes
train_df['phylum_encoded'] = phylum_encoder.transform(train_df['phylum'])
val_df['phylum_encoded'] = phylum_encoder.transform(val_df['phylum'])
test_df['phylum_encoded'] = phylum_encoder.transform(test_df['phylum'])

# Create ensemble generators
ensemble_train_gen = EnsembleDataGenerator(
    dataframe=train_df,
    directory=directory,
    image_size=image_size,
    batch_size=32,
    class_indices=train_generator.class_indices,
    augment=True,
    shuffle=True
)

ensemble_val_gen = EnsembleDataGenerator(
    dataframe=val_df,
    directory=directory,
    image_size=image_size,
    batch_size=32,
    class_indices=train_generator.class_indices,
    augment=False,
    shuffle=False
)

ensemble_test_gen = EnsembleDataGenerator(
    dataframe=test_df,
    directory=directory,
    image_size=image_size,
    batch_size=32,
    class_indices=train_generator.class_indices,
    augment=False,
    shuffle=False
)

# Callbacks for ensemble
ensemble_callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3,
        patience=3,
        min_lr=1e-7,
        verbose=1
    )
]

# Train ensemble model
print("\nTraining Ensemble Model...")
history_ensemble = ensemble_model.fit(
    ensemble_train_gen,
    epochs=50,
    class_weight=class_weight_dict,
    validation_data=ensemble_val_gen,
    callbacks=ensemble_callbacks,
    verbose=1
)

# Plot results
plot_model_results(history_ensemble, 'loss')
plot_model_results(history_ensemble, 'accuracy')

In [ ]:
print("\n" + "="*60)
print("MODEL EVALUATION")
print("="*60)

# Evaluate Image Model
print("\n--- Image Model ---")
image_results = image_model.evaluate(test_generator, verbose=1)
print(f"Test Loss: {image_results[0]:.4f}")
print(f"Test Accuracy: {image_results[1]:.4f}")

# Get predictions for F1 score
y_pred_image = image_model.predict(test_generator, verbose=1)
y_pred_image_classes = np.argmax(y_pred_image, axis=1)
y_true = test_generator.classes

f1_image = f1_score(y_true, y_pred_image_classes, average='weighted')
print(f"F1 Score (Weighted): {f1_image:.4f}")

# Evaluate Metadata Model
print("\n--- Metadata Model ---")
meta_results = metadata_model.evaluate(test_meta, test_labels, verbose=1)
print(f"Test Loss: {meta_results[0]:.4f}")
print(f"Test Accuracy: {meta_results[1]:.4f}")

y_pred_meta = metadata_model.predict(test_meta, verbose=1)
y_pred_meta_classes = np.argmax(y_pred_meta, axis=1)

f1_meta = f1_score(test_labels, y_pred_meta_classes, average='weighted')
print(f"F1 Score (Weighted): {f1_meta:.4f}")

# Evaluate Ensemble Model
print("\n--- Ensemble Model ---")
ensemble_results = ensemble_model.evaluate(ensemble_test_gen, verbose=1)
print(f"Test Loss: {ensemble_results[0]:.4f}")
print(f"Test Accuracy: {ensemble_results[1]:.4f}")

y_pred_ensemble = ensemble_model.predict(ensemble_test_gen, verbose=1)
y_pred_ensemble_classes = np.argmax(y_pred_ensemble, axis=1)

# Get true labels from generator
y_true_ensemble = []
for i in range(len(ensemble_test_gen)):
    _, labels = ensemble_test_gen[i]
    y_true_ensemble.extend(labels)
y_true_ensemble = np.array(y_true_ensemble)

f1_ensemble = f1_score(y_true_ensemble, y_pred_ensemble_classes, average='weighted')
print(f"F1 Score (Weighted): {f1_ensemble:.4f}")

In [ ]:
print("\n" + "="*60)
print("MODEL COMPARISON SUMMARY")
print("="*60)

results_df = pd.DataFrame({
    'Model': ['Image Only', 'Metadata Only', 'Ensemble'],
    'Test Accuracy': [image_results[1], meta_results[1], ensemble_results[1]],
    'Test Loss': [image_results[0], meta_results[0], ensemble_results[0]],
    'F1 Score': [f1_image, f1_meta, f1_ensemble]
})

print(results_df.to_string(index=False))

# Bar plot comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Accuracy
axes[0].bar(results_df['Model'], results_df['Test Accuracy'], color=['#3498db', '#2ecc71', '#e74c3c'])
axes[0].set_title('Test Accuracy', fontsize=14, fontweight='bold')
axes[0].set_ylim(0, 1)
for i, v in enumerate(results_df['Test Accuracy']):
    axes[0].text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

# Loss
axes[1].bar(results_df['Model'], results_df['Test Loss'], color=['#3498db', '#2ecc71', '#e74c3c'])
axes[1].set_title('Test Loss', fontsize=14, fontweight='bold')
for i, v in enumerate(results_df['Test Loss']):
    axes[1].text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

# F1 Score
axes[2].bar(results_df['Model'], results_df['F1 Score'], color=['#3498db', '#2ecc71', '#e74c3c'])
axes[2].set_title('F1 Score (Weighted)', fontsize=14, fontweight='bold')
axes[2].set_ylim(0, 1)
for i, v in enumerate(results_df['F1 Score']):
    axes[2].text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

plt.suptitle('Model Comparison', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Find best model
best_model_idx = results_df['F1 Score'].idxmax()
best_model_name = results_df.loc[best_model_idx, 'Model']
print(f"\nBest Model: {best_model_name}")

# Use predictions from best model
if best_model_name == 'Image Only':
    y_pred_best = y_pred_image_classes
    y_true_best = y_true
elif best_model_name == 'Metadata Only':
    y_pred_best = y_pred_meta_classes
    y_true_best = test_labels
else:
    y_pred_best = y_pred_ensemble_classes
    y_true_best = y_true_ensemble

# Confusion matrix
cm = confusion_matrix(y_true_best, y_pred_best)

plt.figure(figsize=(20, 16))
sns.heatmap(cm, annot=False, fmt='d', cmap='Blues', cbar=True)
plt.title(f'Confusion Matrix: {best_model_name}', fontsize=16, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

print(f"\nConfusion Matrix Shape: {cm.shape}")
print(f"Total Predictions: {cm.sum()}")

### Small custom model

In [ ]:
# Get number of classes
num_classes = len(train_generator.class_indices)
print(f"Number of classes: {num_classes}")

### Transfer learning with keras base model

#### EfficientNetB4

In [ ]:
from tensorflow.keras.applications import EfficientNetB4

In [ ]:
from focal_loss import SparseCategoricalFocalLoss

print("✅ Focal loss imported - will be used for training")

In [ ]:
# from focal_loss import SparseCategoricalFocalLoss

In [ ]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)
class_weight_dict = dict(enumerate(class_weights))

In [ ]:
# Build the classifier with DEEPER HEAD and skip connections
from tensorflow.keras.layers import Add, Activation

x = base_model.output
x = GlobalAveragePooling2D()(x)

# Dense block 1: 1024 neurons
x1 = Dense(1024, kernel_regularizer=L2(0.01))(x)
x1 = BatchNormalization()(x1)
x1 = Activation('relu')(x1)
x1 = Dropout(0.4)(x1)

# Dense block 2: 512 neurons
x2 = Dense(512, kernel_regularizer=L2(0.01))(x1)
x2 = BatchNormalization()(x2)
x2 = Activation('relu')(x2)
x2 = Dropout(0.3)(x2)

# Skip connection (residual connection from pooled features)
x_skip = Dense(512)(x)
x2_combined = Add()([x2, x_skip])

# Output layer (dtype='float32' for mixed precision stability)
output = Dense(N_CLASSES, activation='softmax', dtype='float32')(x2_combined)

# Create model
model = Model(inputs=base_model.input, outputs=output)

print("✅ Model built with deep classification head (3 layers + skip connection)")

In [ ]:
# Compile model with FOCAL LOSS for class imbalance
from tensorflow.keras.callbacks import ModelCheckpoint, TensorBoard

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss=SparseCategoricalFocalLoss(gamma=2.0),  # Focal loss focuses on hard examples!
    metrics=['accuracy'],
)

# Enhanced callbacks with F1 tracking, TensorBoard, and model checkpointing
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=10,  # Increased patience
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,  # More aggressive reduction
        patience=5,
        min_lr=1e-7,
        verbose=1
    ),
    ModelCheckpoint(
        'best_model_stage1.h5',
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    ),
    F1Callback(validation_data=val_generator),
    TensorBoard(
        log_dir=f'logs/{datetime.datetime.now().strftime("%Y%m%d-%H%M%S")}',
        histogram_freq=1
    )
]

print("="*70)
print("STAGE 1: Training classification head only (base model frozen)")
print("="*70)

# Fit the model
history1 = model.fit(
    train_generator,
    epochs=10,  # Stage 1: head only
    class_weight=class_weight_dict,
    validation_data=val_generator,
    callbacks=callbacks,
    verbose=1
)

print(f"\n✅ Stage 1 complete! Best val_loss: {min(history1.history['val_loss']):.4f}")

In [ ]:
# PROGRESSIVE UNFREEZING - Stages 2, 3, 4
# This gradually fine-tunes the pretrained model from top to bottom

print("="*70)
print("STAGE 2: Unfreezing top 50 layers")
print("="*70)

# Unfreeze top 50 layers
base_model.trainable = True
for layer in base_model.layers[:-50]:
    layer.trainable = False

trainable_params = sum([tf.size(w).numpy() for w in model.trainable_weights])
print(f"Trainable parameters: {trainable_params:,}")

model.compile(
    optimizer=Adam(learning_rate=1e-4),  # 10x lower learning rate
    loss=SparseCategoricalFocalLoss(gamma=2.0),
    metrics=['accuracy']
)

history2 = model.fit(
    train_generator,
    epochs=10,
    class_weight=class_weight_dict,
    validation_data=val_generator,
    callbacks=callbacks,
    verbose=1
)

print(f"✅ Stage 2 complete! Best val_loss: {min(history2.history['val_loss']):.4f}")

# ==================== STAGE 3 ====================
print("\n" + "="*70)
print("STAGE 3: Unfreezing top 100 layers")
print("="*70)

# Unfreeze top 100 layers
for layer in base_model.layers[:-100]:
    layer.trainable = False
for layer in base_model.layers[-100:]:
    layer.trainable = True

trainable_params = sum([tf.size(w).numpy() for w in model.trainable_weights])
print(f"Trainable parameters: {trainable_params:,}")

model.compile(
    optimizer=Adam(learning_rate=5e-5),  # Even lower learning rate
    loss=SparseCategoricalFocalLoss(gamma=2.0),
    metrics=['accuracy']
)

history3 = model.fit(
    train_generator,
    epochs=10,
    class_weight=class_weight_dict,
    validation_data=val_generator,
    callbacks=callbacks,
    verbose=1
)

print(f"✅ Stage 3 complete! Best val_loss: {min(history3.history['val_loss']):.4f}")

# ==================== STAGE 4 ====================
print("\n" + "="*70)
print("STAGE 4: Full fine-tuning (all layers unfrozen)")
print("="*70)

# Unfreeze all layers
base_model.trainable = True

trainable_params = sum([tf.size(w).numpy() for w in model.trainable_weights])
print(f"Trainable parameters: {trainable_params:,}")

model.compile(
    optimizer=Adam(learning_rate=1e-5),  # Very low learning rate for full fine-tuning
    loss=SparseCategoricalFocalLoss(gamma=2.0),
    metrics=['accuracy']
)

history4 = model.fit(
    train_generator,
    epochs=10,
    class_weight=class_weight_dict,
    validation_data=val_generator,
    callbacks=callbacks,
    verbose=1
)

print(f"✅ Stage 4 complete! Best val_loss: {min(history4.history['val_loss']):.4f}")

# Save final model
model.save('final_model_progressive_unfreezing.h5')
print("\n🎉 Progressive unfreezing complete! Model saved as 'final_model_progressive_unfreezing.h5'")

In [ ]:
# Compile with categorical crossentropy for multi-class
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

# Set up callbacks
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3,
        patience=3,
        min_lr=1e-7,
        verbose=1
    )
]

# Fit the model (batch_size removed - using generator's batch_size of 32)
history1 = model.fit(
    train_generator,
    epochs=40,
    class_weight=class_weight_dict,
    validation_data=val_generator,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# # Fine-tuning the model
# # Unfreeze base model
# base_model.trainable = True

# # Recompile with much lower learning rate
# model.compile(
#     optimizer=Adam(learning_rate=1e-5),  # 100x lower learning rate
#     loss='categorical_crossentropy',
#     metrics=['accuracy', 'top_k_categorical_accuracy']
# )

# history2 = model.fit(
#     train_generator,
#     epochs=20,
#     validation_data=val_generator,
#     callbacks=callbacks,
#     verbose=1
# )

#### EfficientNetB0

In [ ]:
from tensorflow.keras.applications import EfficientNetB0

In [ ]:
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(*image_size, 3) 
)

# Freeze base model initially
base_model.trainable = False

In [ ]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)
class_weight_dict = dict(enumerate(class_weights))

In [ ]:
# Build the classifier
with tf.device('/GPU:0'):   # Move the model to gpu
    model = Sequential([
        base_model,
        GlobalAveragePooling2D(),
        Dropout(0.4),
        BatchNormalization(),
        Dense(N_CLASSES, kernel_regularizer=L2(0.02), activation='softmax')
    ])

In [ ]:
# See the parameter counts
print("Total params:", model.count_params())
print("Trainable params:", sum([tf.size(w).numpy() for w in model.trainable_weights]))
print("Non-trainable params:", sum([tf.size(w).numpy() for w in model.non_trainable_weights]))

## Results

In [ ]:
# COMPREHENSIVE EVALUATION FUNCTION
def comprehensive_evaluation(model, test_generator):
    """
    Generate detailed evaluation report with insights for the project
    """
    
    # Get predictions
    print("="*70)
    print("GENERATING PREDICTIONS...")
    print("="*70)
    y_pred_probs = model.predict(test_generator, verbose=1)
    y_pred = np.argmax(y_pred_probs, axis=1)
    y_true = test_generator.classes
    class_names = list(test_generator.class_indices.keys())
    
    # ==================== OVERALL METRICS ====================
    print("\n" + "="*70)
    print("OVERALL METRICS")
    print("="*70)
    accuracy = (y_pred == y_true).mean()
    f1_macro = f1_score(y_true, y_pred, average='macro')
    f1_weighted = f1_score(y_true, y_pred, average='weighted')
    f1_micro = f1_score(y_true, y_pred, average='micro')
    
    print(f"Accuracy:     {accuracy:.4f}")
    print(f"F1 Macro:     {f1_macro:.4f}")
    print(f"F1 Weighted:  {f1_weighted:.4f}")
    print(f"F1 Micro:     {f1_micro:.4f}")
    
    # ==================== PER-CLASS REPORT ====================
    print("\n" + "="*70)
    print("PER-CLASS CLASSIFICATION REPORT")
    print("="*70)
    report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
    report_df = pd.DataFrame(report).transpose()
    
    # Best performing classes
    print("\n📊 TOP 10 BEST PERFORMING CLASSES:")
    best_classes = report_df.iloc[:-3].sort_values('f1-score', ascending=False).head(10)
    print(best_classes[['precision', 'recall', 'f1-score', 'support']])
    
    # Worst performing classes
    print("\n❌ TOP 10 WORST PERFORMING CLASSES:")
    worst_classes = report_df.iloc[:-3].sort_values('f1-score', ascending=True).head(10)
    print(worst_classes[['precision', 'recall', 'f1-score', 'support']])
    
    # ==================== CONFIDENCE ANALYSIS ====================
    print("\n" + "="*70)
    print("CONFIDENCE ANALYSIS")
    print("="*70)
    max_probs = np.max(y_pred_probs, axis=1)
    correct_mask = (y_pred == y_true)
    
    print(f"Avg confidence (correct):   {max_probs[correct_mask].mean():.4f}")
    print(f"Avg confidence (incorrect): {max_probs[~correct_mask].mean():.4f}")
    print(f"Confidence gap: {max_probs[correct_mask].mean() - max_probs[~correct_mask].mean():.4f}")
    
    # Visualize confidence distributions
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Confidence distribution
    axes[0].hist(max_probs[correct_mask], bins=50, alpha=0.7, label='Correct', color='green', edgecolor='black')
    axes[0].hist(max_probs[~correct_mask], bins=50, alpha=0.7, label='Incorrect', color='red', edgecolor='black')
    axes[0].set_xlabel('Confidence', fontsize=12)
    axes[0].set_ylabel('Count', fontsize=12)
    axes[0].set_title('Confidence Distribution', fontsize=14, fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Plot 2: Accuracy vs confidence threshold
    thresholds = np.linspace(0, 1, 20)
    accs = []
    coverage = []
    for t in thresholds:
        mask = max_probs >= t
        coverage.append(mask.sum() / len(mask) * 100)  # Percentage
        if mask.sum() > 0:
            accs.append((y_pred[mask] == y_true[mask]).mean() * 100)  # Percentage
        else:
            accs.append(0)
    
    ax2 = axes[1]
    line1 = ax2.plot(thresholds, accs, 'b-', linewidth=2, label='Accuracy', marker='o')
    ax2.set_xlabel('Confidence Threshold', fontsize=12)
    ax2.set_ylabel('Accuracy (%)', color='b', fontsize=12)
    ax2.tick_params(axis='y', labelcolor='b')
    ax2.grid(True, alpha=0.3)
    
    ax2_twin = ax2.twinx()
    line2 = ax2_twin.plot(thresholds, coverage, 'r--', linewidth=2, label='Coverage', marker='s')
    ax2_twin.set_ylabel('Coverage (%)', color='r', fontsize=12)
    ax2_twin.tick_params(axis='y', labelcolor='r')
    
    axes[1].set_title('Accuracy vs Confidence Threshold', fontsize=14, fontweight='bold')
    
    # Add combined legend
    lines = line1 + line2
    labels = [l.get_label() for l in lines]
    ax2.legend(lines, labels, loc='center right')
    
    plt.tight_layout()
    plt.savefig('confidence_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # ==================== TOP CONFUSIONS ====================
    print("\n" + "="*70)
    print("TOP 10 MOST COMMON MISCLASSIFICATIONS")
    print("="*70)
    cm = confusion_matrix(y_true, y_pred)
    cm_no_diag = cm.copy()
    np.fill_diagonal(cm_no_diag, 0)
    
    top_confusions = []
    for i in range(10):
        max_idx = np.unravel_index(np.argmax(cm_no_diag), cm_no_diag.shape)
        count = cm_no_diag[max_idx]
        if count == 0:
            break
        true_class = class_names[max_idx[0]]
        pred_class = class_names[max_idx[1]]
        print(f"{i+1:2d}. {true_class:25s} → {pred_class:25s}: {count:3d} times")
        top_confusions.append((true_class, pred_class, count))
        cm_no_diag[max_idx] = 0
    
    print("\n" + "="*70)
    print("✅ COMPREHENSIVE EVALUATION COMPLETE")
    print("="*70)
    
    return report_df, top_confusions, y_pred, y_pred_probs

# Run comprehensive evaluation
class_names = list(test_generator.class_indices.keys())
results_df, confusions, y_pred_classes, y_pred_probs = comprehensive_evaluation(model, test_generator)

In [ ]:
# Generate Grad-CAM visualizations for correct and incorrect predictions
print("="*70)
print("GRAD-CAM VISUALIZATIONS")
print("="*70)

# Get y_true for comparison
y_true = test_generator.classes

# Get indices of correct and incorrect predictions
correct_indices = np.where(y_pred_classes == y_true)[0]
incorrect_indices = np.where(y_pred_classes != y_true)[0]

# Show examples of CORRECT predictions
print("\n✅ EXAMPLES OF CORRECT PREDICTIONS WITH GRAD-CAM:")
print("-" * 70)
for i in range(min(3, len(correct_indices))):
    idx = correct_indices[i]
    img_path = directory + test_df.iloc[idx]['file_path']
    true_label = class_names[y_true[idx]]
    print(f"\nExample {i+1}: {true_label}")
    display_gradcam(model, img_path, class_names, true_label)

# Show examples of INCORRECT predictions
print("\n❌ EXAMPLES OF INCORRECT PREDICTIONS WITH GRAD-CAM:")
print("-" * 70)
for i in range(min(3, len(incorrect_indices))):
    idx = incorrect_indices[i]
    img_path = directory + test_df.iloc[idx]['file_path']
    true_label = class_names[y_true[idx]]
    print(f"\nExample {i+1}: True = {true_label}")
    display_gradcam(model, img_path, class_names, true_label)

print("\n" + "="*70)
print("✅ GRAD-CAM VISUALIZATION COMPLETE")
print("="*70)

In [ ]:
# GRAD-CAM VISUALIZATION - Show where the model is looking
def make_gradcam(model, img_array, pred_index=None):
    """
    Generate Grad-CAM heatmap showing which parts of the image the model focuses on
    """
    # Find last convolutional layer
    last_conv_layer = None
    for layer in reversed(model.layers):
        if 'conv' in layer.name.lower() and len(layer.output_shape) == 4:
            last_conv_layer = layer.name
            break
    
    if last_conv_layer is None:
        # For EfficientNet, try to find by checking the base model
        for layer in reversed(model.layers):
            if hasattr(layer, 'layers'):  # It's a base model
                for sub_layer in reversed(layer.layers):
                    if 'conv' in sub_layer.name.lower() and len(sub_layer.output_shape) == 4:
                        last_conv_layer = sub_layer.name
                        break
                if last_conv_layer:
                    break
    
    if last_conv_layer is None:
        print("⚠️ No convolutional layer found")
        return None
    
    print(f"Using layer: {last_conv_layer}")
    
    # Create gradient model
    try:
        grad_model = tf.keras.models.Model(
            [model.inputs],
            [model.get_layer(last_conv_layer).output, model.output]
        )
    except:
        # If that fails, try accessing through base_model
        base_layer = model.get_layer('efficientnetb4')
        grad_model = tf.keras.models.Model(
            [model.inputs],
            [base_layer.get_layer(last_conv_layer).output, model.output]
        )
    
    # Compute gradients
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]
    
    # Get gradients
    grads = tape.gradient(class_channel, conv_outputs)
    
    # Global average pooling
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    
    # Multiply
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    
    # Normalize
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-10)
    
    return heatmap.numpy()

def display_gradcam(model, img_path, class_names, true_label=None):
    """Display image with Grad-CAM overlay"""
    # Load and preprocess
    img = tf.keras.preprocessing.image.load_img(img_path, target_size=(500, 375))
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    img_array = preprocess_input(img_array)
    img_array_batch = np.expand_dims(img_array, axis=0)
    
    # Predict
    pred = model.predict(img_array_batch, verbose=0)
    pred_class_idx = np.argmax(pred[0])
    pred_prob = pred[0][pred_class_idx]
    pred_class_name = class_names[pred_class_idx]
    
    # Generate Grad-CAM
    heatmap = make_gradcam(model, img_array_batch, pred_class_idx)
    
    if heatmap is None:
        print("Could not generate Grad-CAM")
        return
    
    # Resize heatmap to image size
    heatmap = cv2.resize(heatmap, (375, 500))
    heatmap = np.uint8(255 * heatmap)
    heatmap_colored = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    
    # Overlay
    img_bgr = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
    superimposed = cv2.addWeighted(img_bgr, 0.6, heatmap_colored, 0.4, 0)
    
    # Display
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    axes[0].imshow(img)
    title = 'Original Image'
    if true_label:
        title += f'\n✅ True: {true_label}'
    axes[0].set_title(title, fontsize=12, fontweight='bold')
    axes[0].axis('off')
    
    axes[1].imshow(heatmap, cmap='jet')
    axes[1].set_title('Grad-CAM\n(Model Attention)', fontsize=12, fontweight='bold')
    axes[1].axis('off')
    
    axes[2].imshow(cv2.cvtColor(superimposed, cv2.COLOR_BGR2RGB))
    title = f'Overlay\n🔮 Pred: {pred_class_name}\nConf: {pred_prob:.3f}'
    axes[2].set_title(title, fontsize=12, fontweight='bold')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()

print("✅ Grad-CAM functions defined")

In [ ]:
plt.plot(history1.history['accuracy'])
plt.plot(history1.history['val_accuracy'])
plt.title('Model Acc')
plt.ylabel('Loss')
plt.xlabel('Epochs')
plt.legend(['train', 'validation'])
plt.show()

In [ ]:
test_results = model.evaluate(test_generator, verbose=1)
print(f"Test Accuracy: {test_results[1]:.4f}")
print(f"Test Loss: {test_results[0]:.4f}")

In [ ]:
# Get predictions for F1 score
y_pred = model.predict(test_generator, verbose=1)
y_pred_classes = np.argmax(y_pred, axis=1)

# Get true labels
y_true = test_generator.classes

# Calculate F1 scores
f1_micro = f1_score(y_true, y_pred_classes, average='micro')
f1_macro = f1_score(y_true, y_pred_classes, average='macro')
f1_weighted = f1_score(y_true, y_pred_classes, average='weighted')

print(f"\nF1 Score (Micro): {f1_micro:.4f}")
print(f"F1 Score (Macro): {f1_macro:.4f}")
print(f"F1 Score (Weighted): {f1_weighted:.4f}")

#Detailed classification report
# print(classification_report(y_true, y_pred_classes, target_names=test_generator.class_indices.keys()))

In [ ]:
# ============================================================================
# FINAL PROJECT SUMMARY
# ============================================================================

print("="*80)
print(" " * 25 + "FINAL PROJECT SUMMARY")
print("="*80)

print("\n📊 DATASET INFORMATION:")
print("-" * 80)
print(f"  Total images (original):          {len(metadata):,}")
print(f"  After outlier removal:            {len(good_images_df):,} ({len(good_images_df)/len(metadata)*100:.1f}%)")
print(f"  Number of classes (families):     {N_CLASSES}")
print(f"  Number of phyla:                  {metadata['phylum'].nunique()}")
print(f"  Train/Val/Test split:             {len(train_df):,} / {len(val_df):,} / {len(test_df):,}")
print(f"  Image size (standardized):        {image_size}")

print("\n🔍 DATA PREPROCESSING:")
print("-" * 80)
print("  Outlier Detection Methods:")
print("    ✓ Distance to centroid (iterative, 2 rounds)")
print("    ✓ ImageNet validation (InceptionV3)")
print("    ✓ Isolation Forest (contamination=0.05)")
print("    ✓ Local Outlier Factor (LOF, n_neighbors=20)")
print("    ✓ DBSCAN clustering (eps=3.0, min_samples=5)")
print(f"  Total outliers removed:           {len(metadata) - len(good_images_df):,} ({(len(metadata) - len(good_images_df))/len(metadata)*100:.1f}%)")
print("\n  Image Augmentation:")
print("    ✓ Rotation (±30°)")
print("    ✓ Width/Height shift (±20%)")
print("    ✓ Horizontal flip")
print("    ✓ Zoom (±25%)")
print("    ✓ Shear (15%)")
print("    ✓ Random brightness (±20%)")
print("    ✓ Random contrast (0.8-1.2)")
print("    ✓ Random saturation (0.8-1.2)")

print("\n🏗️ MODEL ARCHITECTURE:")
print("-" * 80)
print("  Backbone:                         EfficientNetB4 (pretrained on ImageNet)")
print("  Classification Head:")
print("    ├─ GlobalAveragePooling2D")
print("    ├─ Dense(1024) + BatchNorm + ReLU + Dropout(0.4)")
print("    ├─ Dense(512) + BatchNorm + ReLU + Dropout(0.3)")
print("    ├─ Skip connection from pooled features")
print("    └─ Dense(202, softmax, dtype=float32)")
print(f"  Total parameters:                 {model.count_params():,}")
print(f"  Trainable parameters (final):     {sum([tf.size(w).numpy() for w in model.trainable_weights]):,}")

print("\n📈 TRAINING STRATEGY:")
print("-" * 80)
print("  Loss Function:                    Sparse Categorical Focal Loss (gamma=2.0)")
print("  Optimizer:                        Adam")
print("  Class Weighting:                  Balanced (computed from training set)")
print("  Mixed Precision:                  Enabled (float16)")
print("  XLA Compilation:                  Enabled")
print("\n  Progressive Unfreezing (4 stages):")
print("    Stage 1: Head only           | LR: 1e-3  | Epochs: 10")
print("    Stage 2: Top 50 layers       | LR: 1e-4  | Epochs: 10")
print("    Stage 3: Top 100 layers      | LR: 5e-5  | Epochs: 10")
print("    Stage 4: All layers          | LR: 1e-5  | Epochs: 10")
print(f"  Total training epochs:            {len(history1.history['loss']) + len(history2.history['loss']) + len(history3.history['loss']) + len(history4.history['loss'])}")

print("\n📊 CALLBACKS:")
print("-" * 80)
print("  ✓ EarlyStopping (patience=10, restore_best_weights)")
print("  ✓ ReduceLROnPlateau (factor=0.5, patience=5)")
print("  ✓ ModelCheckpoint (save best model)")
print("  ✓ F1 Score tracking (weighted & macro)")
print("  ✓ TensorBoard logging (with histograms)")

print("\n✅ FINAL RESULTS:")
print("-" * 80)
# Get final metrics from comprehensive evaluation
final_acc = results_df.loc['accuracy', 'precision']  # Accuracy is in the precision column for 'accuracy' row
final_f1_weighted = results_df.loc['weighted avg', 'f1-score']
final_f1_macro = results_df.loc['macro avg', 'f1-score']

print(f"  Test Accuracy:                    {final_acc:.4f} ({final_acc*100:.2f}%)")
print(f"  F1 Score (Weighted):              {final_f1_weighted:.4f}")
print(f"  F1 Score (Macro):                 {final_f1_macro:.4f}")
print(f"  F1 Score (Micro):                 {results_df.loc['accuracy', 'recall']:.4f}")  # Micro F1 = Accuracy

print("\n🔬 ANALYSIS COMPLETED:")
print("-" * 80)
print("  ✓ Comprehensive per-class metrics (all 202 classes)")
print("  ✓ Confusion matrix analysis")
print("  ✓ Confidence calibration analysis")
print("  ✓ Top 10 best/worst performing classes")
print("  ✓ Top 10 most common misclassifications")
print("  ✓ Grad-CAM visualizations (correct & incorrect predictions)")

print("\n💡 INNOVATIONS IMPLEMENTED:")
print("-" * 80)
print("  ✓ Multi-method outlier detection pipeline (5 complementary techniques)")
print("  ✓ Deep classification head with skip connections")
print("  ✓ Progressive unfreezing strategy (4-stage)")
print("  ✓ Focal loss for class imbalance")
print("  ✓ Enhanced data augmentation (geometric + color)")
print("  ✓ F1 score tracking during training")
print("  ✓ Grad-CAM interpretability")
print("  ✓ Confidence calibration analysis")

print("\n📁 FILES GENERATED:")
print("-" * 80)
print("  ✓ best_model_stage1.h5            (Best model from stage 1)")
print("  ✓ final_model_progressive_unfreezing.h5  (Final trained model)")
print("  ✓ confidence_analysis.png         (Confidence distribution plots)")
print("  ✓ logs/                           (TensorBoard training logs)")

print("\n" + "="*80)
print(" " * 22 + "🎉 PROJECT COMPLETE! 🎉")
print("="*80)
print("\nModel is ready for evaluation in the report!")
print("Expected improvements over baseline:")
print(f"  • Baseline (linear classifier):   ~79% accuracy")
print(f"  • Current model:                  ~{final_acc*100:.0f}% accuracy (expected: 85-90%)")
print(f"  • Improvement:                    +{(final_acc - 0.79)*100:.0f} percentage points")
print("="*80)

In [ ]:
# Create confusion matrix
cm = confusion_matrix(y_true, y_pred_classes)

# Plot confusion matrix
plt.figure(figsize=(20, 16))
sns.heatmap(cm, annot=False, fmt='d', cmap='Blues', cbar=True)         # annot=np.where(cm > 0, cm, ""), annot_kws={"size": 5}
plt.title('Confusion Matrix: Predictions vs Actual Labels')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

# Print shape and summary
print(f"Confusion Matrix Shape: {cm.shape}")
print(f"Total Predictions: {cm.sum()}")

In [ ]:
# y_pred_classes[1]
dict(test_generator.class_indices.items())[low_images_class]

In [ ]:
def class_metrics(class_name):

    class_idx = test_generator.class_indices[class_name]

    TP = int(cm[class_idx, class_idx])
    FN = int(cm[class_idx, :].sum() - TP)   # actual positives missed
    FP = int(cm[:, class_idx].sum() - TP)   # predicted as this class but not actual
    TN = int(cm.sum() - (TP + FP + FN))
    support = TP + FN
    
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0   #per-class accuracy / sensitivity
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    class_accuracy = (TP + TN) / cm.sum()               # accuracy treating this class vs rest
    
    # class_name =idx_to_class[class_idx]
    print(f"Class: {class_name} (index={class_idx})")
    print(f" Support (true samples) : {support}")
    print(f" TP: {TP} | FN: {FN} | FP: {FP} | TN: {TN}")
    print(f" Precision : {precision:.4f}")
    print(f" Recall    : {recall:.4f}")     
    print(f" F1-score  : {f1:.4f}")
    print(f" Class-wise accuracy : {class_accuracy:.4f}")



In [ ]:
class_metrics(low_images_class)